In [21]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from pathlib import Path
import os

os.chdir(r"c:\Users\Huy\Documents\UIT\Sem_4\DS102")

FEATURES = 'data/final/features.csv'
TARGET = 'data/final/target.csv'

LABELS = [
    'SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON',
    'DELTA_LAT_6h', 'DELTA_LON_6h'
]

DROPS = ['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR'] + [
        col for col in [f'DELTA_LAT_{i}h' for i in [6, 12, 18, 24]]] + [
        col for col in [f'DELTA_LON_{i}h' for i in [6, 12, 18, 24]]]

GDBT_6H = {
    'n_estimators': 1000,       # Số lượng cây quyết định
    'max_depth': 3,             # Độ sâu tối đa của mỗi cây
    'max_features': 60,         # Lấy ngẫu nhiên 60 biến (trong số 88+ biến) tại mỗi lần phân nhánh
    'learning_rate': 0.1,       # Tốc độ học
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'loss': 'squared_error',
    'random_state': 42
}

def haversine_distance(lat_true, lon_true, lat_pred, lon_pred):
    """Tính khoảng cách thực tế (km) trên mặt cầu Trái Đất"""
    lat_true, lon_true, lat_pred, lon_pred = map(np.radians, [lat_true, lon_true, lat_pred, lon_pred])
    dlat = lat_pred - lat_true
    dlon = lon_pred - lon_true
    a = np.sin(dlat/2.0)**2 + np.cos(lat_true) * np.cos(lat_pred) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # Bán kính Trái Đất ~ 6371 km

def prepare_data():
    features = pd.read_csv(FEATURES)
    target = pd.read_csv(TARGET)

    df = features.merge(target, on=['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON'], how='left')
    df = df.dropna(subset=['DELTA_LAT_6h', 'DELTA_LON_6h'])
    train_df = df[(df['YEAR'] >= 2000) & (df['YEAR'] <= 2020)]  # 21 years
    val_df = df[(df['YEAR'] >= 2021) & (df['YEAR'] <= 2022)]    # 2 years
    test_df = df[(df['YEAR'] >= 2023) & (df['YEAR'] <= 2024)]   # 2 years

    return train_df, val_df, test_df

In [22]:
train_df, val_df, test_df = prepare_data()
X_train = train_df.drop(columns=DROPS)
y_train_lat = train_df['DELTA_LAT_6h']
y_train_lon = train_df['DELTA_LON_6h']

X_val = val_df.drop(columns=DROPS)
y_val_lat = val_df['DELTA_LAT_6h']
y_val_lon = val_df['DELTA_LON_6h']

X_test = test_df.drop(columns=DROPS)
y_test_lat = test_df['DELTA_LAT_6h']
y_test_lon = test_df['DELTA_LON_6h']

current_lat_test = test_df['LAT'].values
current_lon_test = test_df['LON'].values

print("Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...")
model_lat = GradientBoostingRegressor(**GDBT_6H)
model_lat.fit(X_train, y_train_lat)

print("Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...")
model_lon = GradientBoostingRegressor(**GDBT_6H)
model_lon.fit(X_train, y_train_lon)

pred_delta_lat = model_lat.predict(X_test)
pred_delta_lon = model_lon.predict(X_test)

pred_lat_6h = current_lat_test + pred_delta_lat
pred_lon_6h = current_lon_test + pred_delta_lon

true_lat_6h = current_lat_test + y_test_lat.values
true_lon_6h = current_lon_test + y_test_lon.values

test_errors_km = haversine_distance(true_lat_6h, true_lon_6h, pred_lat_6h, pred_lon_6h)
mean_error = np.mean(test_errors_km)

print(f"\n=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 6H TRUNG BÌNH (Mean Forecast Error): {mean_error:.2f} km")

Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...
Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...

=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 6H TRUNG BÌNH (Mean Forecast Error): 31.86 km
